# YoVibe ML Training Notebook

## Complete Training Pipeline for Venue Vibe Classification Model

This notebook provides a complete end-to-end ML training pipeline for the YoVibe venue vibe classification model. It includes:

1. Google Drive mounting and environment setup
2. Dataset loading from pre-augmented images
3. Model building (Simple CNN - TensorFlow.js compatible)
4. Two model training: Classification and Regression
5. Model evaluation
6. Model saving in TensorFlow.js format
8. Download links and verification

## Step 1: Mount Google Drive and Setup Environment

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/gdrive')

# Check Google Drive contents
import os
print("Google Drive mounted successfully!")
print("\nContents of /content/gdrive/MyDrive:")
for item in os.listdir('/content/gdrive/MyDrive'):
    print(f"  - {item}")

## Step 2: Install Required Dependencies

In [ ]:
# Install TensorFlow and other required packages
# Use TensorFlow 2.x for Keras 2 compatibility with TensorFlow.js
!pip install tensorflow==2.15.0 numpy pillow matplotlib seaborn scikit-learn -q

import tensorflow as tf
print(f"TensorFlow version: {tf.__version__}")
print(f"Keras version: {tf.keras.__version__}")

## Step 3: Load Dataset from Google Drive

In [ ]:
import os
import json
import numpy as np
from pathlib import Path

# Configuration - UPDATE THIS PATH to your dataset location
DATASET_PATH = '/content/gdrive/MyDrive/YoVibe/training-data/images'
METADATA_PATH = os.path.join(DATASET_PATH, 'metadata.json')

# Verify dataset exists
if os.path.exists(DATASET_PATH):
    print(f"✓ Dataset found at: {DATASET_PATH}")
    
    # List rating directories
    rating_dirs = [d for d in os.listdir(os.path.join(DATASET_PATH, 'by-rating')) 
                   if d.startswith('rating_')]
    print(f"\nRating categories found: {len(rating_dirs)}")
    for rating_dir in sorted(rating_dirs):
        rating_path = os.path.join(DATASET_PATH, 'by-rating', rating_dir)
        num_images = len([f for f in os.listdir(rating_path) if f.endswith('.png')])
        print(f"  - {rating_dir}: {num_images} images")
else:
    print(f"✗ Dataset not found at: {DATASET_PATH}")
    print("\nPlease upload your dataset to Google Drive and update DATASET_PATH")

## Step 4: Load Pre-Augmented Dataset

In [ ]:
import numpy as np
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from collections import Counter

# Image parameters
IMG_SIZE = 224
BATCH_SIZE = 32

# Use pre-augmented dataset (already has 1 original + 5 augmented per source)
AUGMENTED_DATASET_PATH = os.path.join(DATASET_PATH, 'by-rating-augmented')

# Count images in augmented dataset
total_images = 0
if os.path.exists(AUGMENTED_DATASET_PATH):
    for rating_dir in os.listdir(AUGMENTED_DATASET_PATH):
        rating_path = os.path.join(AUGMENTED_DATASET_PATH, rating_dir)
        if os.path.isdir(rating_path):
            num_images = len([f for f in os.listdir(rating_path) if f.endswith(('.png', '.jpg', '.jpeg'))])
            total_images += num_images
            print(f"  {rating_dir}: {num_images} images")

print(f"\n✓ Using pre-augmented dataset!")
print(f"  Total images: {total_images}")

In [ ]:
# Create training/validation generators from the augmented dataset
# Note: We use Rescaling in the model itself, so no preprocessing needed here
train_datagen = ImageDataGenerator(
    validation_split=0.2
)

val_datagen = ImageDataGenerator(
    validation_split=0.2
)

In [ ]:
# Create training generator
train_generator = train_datagen.flow_from_directory(
    AUGMENTED_DATASET_PATH,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='sparse',
    subset='training',
    shuffle=True
)

# Create validation generator
val_generator = val_datagen.flow_from_directory(
    AUGMENTED_DATASET_PATH,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='sparse',
    subset='validation',
    shuffle=False
)

# Get class names
class_names = list(train_generator.class_indices.keys())
print("Classes:", class_names)
print("Training samples:", train_generator.samples)
print("Validation samples:", val_generator.samples)

## Step 5: Calculate Class Weights

In [ ]:
# Analyze class distribution for balanced training
class_counts = Counter(train_generator.classes)
print("Class Distribution:")
for class_name, count in sorted(class_counts.items()):
    print(f"  {class_names[class_name]}: {count}")

# Calculate class weights
total_samples = sum(class_counts.values())
num_classes = len(class_counts)
class_weights = {}
for i in range(num_classes):
    class_weights[i] = total_samples / (num_classes * class_counts.get(i, 1))

print("\nClass Weights for Balanced Training:")
for class_name, weight in sorted(class_weights.items(), key=lambda x: x[0]):
    print(f"  {class_names[class_name]}: {weight:.2f}")

## Step 6a: Build Classification Model CNN

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, Input, Rescaling
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau

# Model parameters
NUM_CLASSES = len(class_names)
IMG_SIZE = 224
INPUT_SHAPE = (IMG_SIZE, IMG_SIZE, 3)

def create_classification_model():
    model = Sequential([
        Input(shape=INPUT_SHAPE),
        Rescaling(1./255),
        Conv2D(32, (3, 3), activation='relu', padding='same'),
        MaxPooling2D((2, 2)),
        Conv2D(64, (3, 3), activation='relu', padding='same'),
        MaxPooling2D((2, 2)),
        Conv2D(128, (3, 3), activation='relu', padding='same'),
        MaxPooling2D((2, 2)),
        Conv2D(256, (3, 3), activation='relu', padding='same'),
        MaxPooling2D((2, 2)),
        Flatten(),
        Dense(256, activation='relu'),
        Dropout(0.5),
        Dense(128, activation='relu'),
        Dropout(0.3),
        Dense(NUM_CLASSES, activation='softmax')
    ])
    return model

# Create classification model
model_cls = create_classification_model()
model_cls.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)
model_cls.summary()
print(f"\nClassification Model: {NUM_CLASSES} classes")

## Step 6b: Build Regression Model CNN

In [ ]:
def create_regression_model():
    model = Sequential([
        Input(shape=INPUT_SHAPE),
        Rescaling(1./255),
        Conv2D(32, (3, 3), activation='relu', padding='same'),
        MaxPooling2D((2, 2)),
        Conv2D(64, (3, 3), activation='relu', padding='same'),
        MaxPooling2D((2, 2)),
        Conv2D(128, (3, 3), activation='relu', padding='same'),
        MaxPooling2D((2, 2)),
        Conv2D(256, (3, 3), activation='relu', padding='same'),
        MaxPooling2D((2, 2)),
        Flatten(),
        Dense(256, activation='relu'),
        Dropout(0.5),
        Dense(128, activation='relu'),
        Dropout(0.3),
        Dense(1, activation='linear')
    ])
    return model

# Create regression model
model_reg = create_regression_model()
model_reg.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='mse',
    metrics=['mae']
)
model_reg.summary()
print(f"\nRegression Model: Single value (0-5)")

## Step 7: Train Both Models

In [ ]:
# Training parameters
EPOCHS = 20
STEPS_PER_EPOCH = train_generator.samples // BATCH_SIZE
VALIDATION_STEPS = val_generator.samples // BATCH_SIZE

# Callbacks for classification model
callbacks_cls = [
    EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6, verbose=1),
    ModelCheckpoint('/content/gdrive/MyDrive/YoVibe/models/best_class_model.keras', monitor='val_accuracy', save_best_only=True, verbose=1)
]

# Callbacks for regression model
callbacks_reg = [
    EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6, verbose=1),
    ModelCheckpoint('/content/gdrive/MyDrive/YoVibe/models/best_reg_model.keras', monitor='val_loss', save_best_only=True, verbose=1)
]

print("="*60)
print("TRAINING BOTH MODELS")
print("="*60)

# Train Classification Model
print("\n--- Training Classification Model ---")
history_cls = model_cls.fit(
    train_generator,
    epochs=EPOCHS,
    steps_per_epoch=STEPS_PER_EPOCH,
    validation_data=val_generator,
    validation_steps=VALIDATION_STEPS,
    class_weight=class_weights,
    callbacks=callbacks_cls,
    verbose=1
)

# Train Regression Model
print("\n--- Training Regression Model ---")
history_reg = model_reg.fit(
    train_generator,
    epochs=EPOCHS,
    steps_per_epoch=STEPS_PER_EPOCH,
    validation_data=val_generator,
    validation_steps=VALIDATION_STEPS,
    callbacks=callbacks_reg,
    verbose=1
)

## Step 8: Fine-Tune with Lower Learning Rate

In [ ]:
# Fine-tune the simple CNN with lower learning rateprint("\nFine-tuning with lower learning rate...")

# Recompile with lower learning ratemodel.compile(    optimizer=Adam(learning_rate=0.0001),    loss='sparse_categorical_crossentropy',    metrics=['accuracy'])
# Continue training
history_finetune = model.fit(
    train_generator,
    epochs=10,
    steps_per_epoch=STEPS_PER_EPOCH,
    validation_data=val_generator,
    validation_steps=VALIDATION_STEPS,
    class_weight=class_weights,
    callbacks=callbacks,
    verbose=1
)

## Step 9: Evaluate Model Performance

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix

# Plot training history
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Accuracy plot
axes[0].plot(history.history['accuracy'], label='Training Accuracy')
axes[0].plot(history.history['val_accuracy'], label='Validation Accuracy')
axes[0].set_title('Model Accuracy')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()
axes[0].grid(True)

# Loss plot
axes[1].plot(history.history['loss'], label='Training Loss')
axes[1].plot(history.history['val_loss'], label='Validation Loss')
axes[1].set_title('Model Loss')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.savefig('/content/gdrive/MyDrive/YoVibe/models/training_history.png', dpi=150)
plt.show()

# Evaluate on validation set
val_loss, val_acc, val_mae = model.evaluate(val_generator)
print(f"\nValidation Results:")
print(f"  Loss: {val_loss:.4f}")
print(f"  Accuracy: {val_acc:.4f}")
print(f"  MAE: {val_mae:.4f}")

## Step 10: Generate Classification Report and Confusion Matrix

In [ ]:
# Get predictions
val_generator.reset()
Y_pred = model.predict(val_generator)
y_pred = np.argmax(Y_pred, axis=1)
y_true = val_generator.classes

# Classification report
print('Classification Report:')
print(classification_report(y_true, y_pred, target_names=class_names))

# Confusion matrix
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=class_names, yticklabels=class_names)
plt.title('Confusion Matrix')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.tight_layout()
plt.savefig('/content/gdrive/MyDrive/YoVibe/models/confusion_matrix.png', dpi=150)
plt.show()

## Step 11: Save the Trained Model in Multiple Formats

In [ ]:
# Create models directory if it doesn't exist
models_dir = '/content/gdrive/MyDrive/YoVibe/models'
os.makedirs(models_dir, exist_ok=True)

# Install tensorflowjs for TensorFlow.js conversion
!pip install tensorflowjs -q

import tensorflowjs as tfjs

# ============================================================
# First: Save model in Keras format
# ============================================================
keras_path = os.path.join(models_dir, 'vibe_model.keras')
model.save(keras_path)
print(f"✓ Model saved in Keras format: {keras_path}")

# ============================================================
# FORMAT 1: TensorFlow.js (FOR WEB APPS)
# This format works with @tensorflow/tfjs in React Web
# ============================================================
tfjs_model_dir = os.path.join(models_dir, 'vibe_model_tfjs')

# Clean up old folder
import shutil
if os.path.exists(tfjs_model_dir):
    shutil.rmtree(tfjs_model_dir)
os.makedirs(tfjs_model_dir)

# Convert using Python API
tfjs.converters.convert_tf_saved_model(
    keras_path,
    tfjs_model_dir
)
print(f"✓ TensorFlow.js model saved: {tfjs_model_dir}")

# List the output files
print("\nTensorFlow.js model files:")
for f in os.listdir(tfjs_model_dir):
    print(f"  - {f}")

In [ ]:
# ============================================================
# FORMAT 2: TensorFlow Lite (FOR REACT NATIVE)
# This format works with react-native-tflite
# ============================================================
tflite_model_path = os.path.join(models_dir, 'vibe_model.tflite')
converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_model = converter.convert()
with open(tflite_model_path, 'wb') as f:
    f.write(tflite_model)
print(f"✓ TensorFlow Lite model saved: {tflite_model_path}")

# ============================================================
# FORMAT 3: Keras H5 (UNIVERSAL BACKUP)
# ============================================================
h5_model_path = os.path.join(models_dir, 'vibe_model.h5')
model.save(h5_model_path)
print(f"✓ Keras H5 model saved: {h5_model_path}")

# ============================================================
# FORMAT 4: SavedModel (TENSORFLOW SERVING)
# ============================================================
savedmodel_path = os.path.join(models_dir, 'vibe_model_savedmodel')
model.save(savedmodel_path)
print(f"✓ SavedModel saved: {savedmodel_path}")

In [ ]:
# Verify saved models
print("\n" + "="*60)
print("MODEL VERIFICATION")
print("="*60)

model_files = {
    'TensorFlow.js': tfjs_model_dir,
    'TensorFlow Lite': tflite_model_path,
    'Keras H5': h5_model_path,
    'SavedModel': savedmodel_path
}

for format_name, file_path in model_files.items():
    if os.path.exists(file_path):
        if os.path.isdir(file_path):
            size = sum(os.path.getsize(os.path.join(dirpath, filename))
                      for dirpath, _, filenames in os.walk(file_path)
                      for filename in filenames)
        else:
            size = os.path.getsize(file_path)
        print(f"\n✓ {format_name}: {file_path}")
        print(f"  Size: {size / (1024*1024):.2f} MB")
    else:
        print(f"\n✗ {format_name}: NOT FOUND")

## Step 12: Test Inference with Saved Model

In [ ]:
# Load and test the saved model
print("Testing inference with saved model...")

# Load Keras model
loaded_model = tf.keras.models.load_model(h5_model_path)

# Test with a sample batch
test_batch, test_labels = next(val_generator)
predictions = loaded_model.predict(test_batch)

print(f"\nTest batch shape: {test_batch.shape}")
print(f"Predictions shape: {predictions.shape}")

# Display sample predictions
print("\nSample Predictions:")
print("-" * 60)
for i in range(min(5, len(predictions))):
    pred_class = np.argmax(predictions[i])
    confidence = predictions[i][pred_class]
    true_class = int(test_labels[i])
    print(f"  Sample {i+1}: Predicted={class_names[pred_class]} "
          f"(confidence: {confidence:.2f}), True={class_names[true_class]}")

## Step 13: Download Links and Usage Instructions

In [ ]:
print("\n" + "="*60)
print("TRAINING COMPLETE - MODEL USAGE INSTRUCTIONS")
print("="*60)

print("\n" + "="*60)
print("FOR REACT WEB APPS (TensorFlow.js)")
print("="*60)
print("\nUse the TensorFlow.js format:")
print("  " + tfjs_model_dir)

In your web app:
```javascript
import * as tf from '@tensorflow/tfjs';
const model = await tf.loadLayersModel('/vibe_model_tfjs/model.json');
const prediction = model.predict(inputTensor);
```

="*60)
print("FOR REACT NATIVE APPS (TensorFlow Lite)")
print("="*60)
print("\nUse the TensorFlow Lite format:")
print("  " + tflite_model_path)
print("\nIn your React Native app, use:")
print("  - react-native-tflite")
print("  - mobilenet (with TFLite support)")


In [ ]:
# Final summary
print("\n" + "="*60)
print("FINAL SUMMARY")
print("="*60)

print("Dataset: " + DATASET_PATH)
print("Source images: " + str(total_source_images))
print("Augmented images: " + str(total_augmented_images))
print("Augmentation factor: " + str(augmentations_per_image + 1) + "x per image")
print("Training samples: " + str(train_generator.samples))
print("Validation samples: " + str(val_generator.samples))
print("Number of classes: " + str(NUM_CLASSES))
print("Classes: " + str(class_names))
print("Image size: " + str(IMG_SIZE) + "x" + str(IMG_SIZE))
print("Final validation accuracy: " + str(val_acc))
print("Final validation loss: " + str(val_loss))
print("Models saved to: " + models_dir)

print("🎉 Training completed successfully!")

print("\n📥 Download your model files from Google Drive:")
print(f"  • TensorFlow.js: {tfjs_model_dir}")
print(f"  • TensorFlow Lite: {tflite_model_path}")
print(f"  • Keras H5: {h5_model_path}")